In [ ]:
import os
import pandas as pd
import cv2
from sklearn.model_selection import train_test_split
import shutil

In [ ]:
IMAGE_DIR = './dataset/images/'
ANNOTATIONS_DIR = './dataset/annotations/'
YOLO_DATASET_DIR = './Yolo_dataset'

In [ ]:
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images/val'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels/train'), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels/val'), exist_ok=True)

In [ ]:
class_names = [d for d in os.listdir(IMAGE_DIR) if os.path.isdir(os.path.join(IMAGE_DIR, d))]
class_names

In [ ]:
csv_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.csv')]
csv_files

In [ ]:
yolo_class_map = {name: i for i, name in enumerate(class_names)}
yolo_class_map

In [ ]:
dataset = []

for i in range(len(class_names)):
    class_name = class_names[i] 
    class_dir = os.path.join(IMAGE_DIR, class_name) 
    csv_file_name = csv_files[i] 
    
    csv_path = os.path.join(ANNOTATIONS_DIR, csv_file_name) 
    df_annotations = pd.read_csv(csv_path) 

    for image_name in os.listdir(class_dir):
        image_path = os.path.join(class_dir, image_name) 
        
        image = cv2.imread(image_path) 
        h, w, _ = image.shape 
        
        ann = df_annotations[df_annotations['image_name'] == image_name].iloc[0,1:].tolist()

        if (ann[2] > ann[0] and ann[3] > ann[1]): 
            x_min, y_min, x_max, y_max = [float(coord) for coord in ann]
            
            x_center = ((x_min + x_max) / 2) / w
            y_center = ((y_min + y_max) / 2) / h
            box_width = (x_max - x_min) / w
            box_height = (y_max - y_min) / h
            
            class_id = yolo_class_map[class_name] 
            yolo_label_content = f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}" 
            
            item_data = {
                'class_name': class_name,
                'image_name': image_name,
                'source_path': image_path,
                'yolo_label': yolo_label_content
            }
            dataset.append(item_data)
        else:
            print(f"️Invalid box found and removed in '{image_path}': {ann}") #

In [ ]:
train_data, val_data = train_test_split(dataset, test_size=0.1, random_state=42, stratify=[d['class_name'] for d in dataset])

In [ ]:
print(f"{len(train_data)}")
print(f"{len(val_data)}")

In [ ]:
def save_yolo_files(data_list, split_name):
    for item in data_list:
        class_name = item['class_name']
        image_name = item['image_name']
        source_image_path = item['source_path']
        yolo_label_content = item['yolo_label']
        
        new_unique_filename = f"{class_name}_{image_name}"
        base_new_filename = os.path.splitext(new_unique_filename)[0]

        dest_image_path = os.path.join(YOLO_DATASET_DIR, 'images', split_name, new_unique_filename)
        dest_label_path = os.path.join(YOLO_DATASET_DIR, 'labels', split_name, base_new_filename + '.txt')

        shutil.copy(source_image_path, dest_image_path)
        with open(dest_label_path, 'w') as f:
            f.write(yolo_label_content)

In [ ]:
save_yolo_files(train_data, 'train')

In [ ]:
save_yolo_files(val_data, 'val')